# 01 — Questions, retrieval and generation

Produces everything that needs a GPU: for every model–dataset pair it stores the closed-book answer, the RAG answer,
token-level uncertainty features, retrieval features φ(x) and the hidden states of four layers. All later steps
(`02_analysis.ipynb` and the scripts) read these files and run on a CPU.

| Order | Model | Datasets |
|---|---|---|
| 1–2 | Qwen2.5-7B, Llama-3.1-8B | TriviaQA |
| 3–4 | Qwen2.5-7B, Llama-3.1-8B | the other six datasets |
| 5 | Mistral-7B-v0.3 | all seven datasets |
| 6–7 | Qwen2.5-3B, Qwen2.5-14B | TriviaQA, NQ, PopQA |

TriviaQA runs first for the two main models because the success rule fixed after the pilot study is checked on it.

**Running.** Use an A100 runtime (high RAM) on Colab and run the cells from top to bottom. Every step writes its
output in small parts, so after a disconnect you only rerun the cells; finished parts are skipped.

**Model access.** Llama and Mistral are gated on Hugging Face. A model without access is skipped with a message
and the queue continues.

## 0. Settings

In [ ]:
PILOT = False                     # True: small pipeline test (Qwen2.5-7B, NQ + PopQA, 200 questions)

DATASETS_ALL = ["triviaqa", "nq", "popqa", "squad", "hotpotqa", "2wikimultihopqa", "musique"]
N_PER_DATASET = 200 if PILOT else 3000     # MuSiQue has about 2,400 test questions; all of them are used
SEED = 2026

QWEN7 = "Qwen/Qwen2.5-7B-Instruct"
LLAMA8 = "meta-llama/Llama-3.1-8B-Instruct"
MISTRAL7 = "mistralai/Mistral-7B-Instruct-v0.3"
QWEN3 = "Qwen/Qwen2.5-3B-Instruct"
QWEN14 = "Qwen/Qwen2.5-14B-Instruct"

FIRST = ["triviaqa"]
REST = [d for d in DATASETS_ALL if d not in FIRST]
SCALE = ["triviaqa", "nq", "popqa"]

RUN_QUEUE = [
    (QWEN7, FIRST),
    (LLAMA8, FIRST),
    (QWEN7, REST),
    (LLAMA8, REST),
    (MISTRAL7, DATASETS_ALL),
    (QWEN3, SCALE),
    (QWEN14, SCALE),
]
if PILOT:
    RUN_QUEUE = [(QWEN7, ["nq", "popqa"])]

GEN_BATCH = 32
GEN_BATCH_BY_MODEL = {QWEN14: 16}  # halved again automatically on an out-of-memory error

# Retrieval
BM25_K = 50
TOP_K = 5
E5_MODEL = "intfloat/e5-base-v2"

# Generation
MAX_NEW_TOKENS = 32
CHUNK = 64
LAYER_FRACS = [0.5, 0.67, 0.83, 0.92]    # relative depth of the stored layers

DRIVE_ROOT = "/content/drive/MyDrive/rarag"   # change to your own folder

DATASETS = list(dict.fromkeys(d for _, ds in RUN_QUEUE for d in ds))
for m, ds in RUN_QUEUE:
    print(f"{m.split('/')[-1]:28s} {ds}")

## 1. Folders

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

RUN_TAG = "pilot" if PILOT else "full"
DIRS = {
    "questions": f"{DRIVE_ROOT}/questions",
    "retrieval": f"{DRIVE_ROOT}/retrieval",
    "cache": f"{DRIVE_ROOT}/cache",
    "logs": f"{DRIVE_ROOT}/logs",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)


def cache_dir(model_name):
    return f"{DIRS['cache']}/{model_name.split('/')[-1]}/{RUN_TAG}"

## 2. Installation

Pyserini needs Java 21. On the first run the cell installs the packages and restarts the runtime (Colab reports a
crash; this is expected). Then run cells 0 and 1 again and this cell once more; the installation is skipped and only
the environment variables are set.

In [ ]:
import os

if not os.path.exists("/content/.rarag_setup_done"):
    !apt-get -qq update && apt-get -qq install -y openjdk-21-jdk-headless > /dev/null
    !pip -q install pyserini faiss-cpu accelerate datasets pyarrow scikit-learn
    !pip -q install -U --force-reinstall --no-deps pillow
    open("/content/.rarag_setup_done", "w").close()
    os.kill(os.getpid(), 9)

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSERINI_CACHE"] = f"{DRIVE_ROOT}/pyserini_cache"   # keeps the BM25 index between sessions
!java -version 2>&1 | head -1

## 3. Hugging Face login

Add your token once as a Colab secret named `HF_TOKEN` (key icon in the left panel) and allow this notebook to read
it. Without the secret the cell asks for the token.

In [ ]:
from huggingface_hub import login, whoami

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()
print("logged in as", whoami()["name"])

## 4. Questions

FlashRAG provides all datasets in one format (`id`, `question`, `golden_answers`). The first available split is used
(test, then dev, then train), shuffled with a fixed seed. The sample is saved once and shared by all models.

In [ ]:
import json
import random

import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "RUC-NLPIR/FlashRAG_datasets"
_files = list_repo_files(REPO, repo_type="dataset")


def pick_split_file(name):
    cands = [f for f in _files if f.startswith(f"{name}/") and f.endswith(".jsonl")]
    for split in ["test", "dev", "val", "validation", "train"]:
        for f in cands:
            if f.split("/")[-1].startswith(split):
                return f
    raise FileNotFoundError(f"{name}: no jsonl file in {cands}")


def load_questions(name):
    out = f"{DIRS['questions']}/{name}_{N_PER_DATASET}_{SEED}.jsonl"
    if os.path.exists(out):
        return pd.read_json(out, lines=True, dtype={"id": str})
    f = pick_split_file(name)
    path = hf_hub_download(REPO, f, repo_type="dataset")
    rows = [json.loads(line) for line in open(path, encoding="utf-8")]
    rows = [{"id": str(r["id"]), "question": r["question"], "golden_answers": r["golden_answers"]} for r in rows]
    random.Random(SEED).shuffle(rows)
    rows = [r for r in rows if r["golden_answers"]][:N_PER_DATASET]
    df = pd.DataFrame(rows)
    df["dataset"], df["source_split"] = name, f
    df.to_json(out, orient="records", lines=True, force_ascii=False)
    return df


QUESTIONS = {name: load_questions(name) for name in DATASETS}
for k, v in QUESTIONS.items():
    print(f"{k:16s} n={len(v):5d}  {v['source_split'].iloc[0]}")

## 5. Correctness label

An answer is correct if a normalised gold answer occurs in it as a whole word sequence (the same rule as
`rarag/labels.py`). Exact match and token F1 are stored as well.

In [ ]:
import re
import string
from collections import Counter

_PUNCT = re.compile("[" + re.escape(string.punctuation) + "]")


def normalize(s):
    s = _PUNCT.sub(" ", str(s).lower())
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def score_answer(pred, golds):
    p = normalize(pred)
    gs = [normalize(g) for g in golds if str(g).strip()]
    acc = float(any(g and f" {g} " in f" {p} " for g in gs))
    em = float(any(p == g for g in gs))
    f1 = 0.0
    for g in gs:
        pt, gt = p.split(), g.split()
        common = sum((Counter(pt) & Counter(gt)).values())
        if common:
            pr, rc = common / len(pt), common / len(gt)
            f1 = max(f1, 2 * pr * rc / (pr + rc))
    return acc, em, f1


assert score_answer("It was Paris, France.", ["Paris"])[:2] == (1.0, 0.0)
assert score_answer("Lillehammer", ["Lille"])[0] == 0.0
assert score_answer("Ashoka's father", ["Ashoka"])[0] == 1.0

## 6. Retrieval: BM25 top-50, e5 re-ranking, top-5

The prebuilt Pyserini index `wikipedia-dpr-100w` is downloaded once to Drive (the download resumes after an
interruption and is checked with MD5), unpacked to the local disk in each session, and queried in parts of 100
questions.

Retrieval features φ(x): top BM25 score, gap between the first two BM25 scores, top e5 similarity, gap between the
first two e5 similarities, mean and standard deviation of the top-5 e5 similarities, and the number of candidates.
The scorers use the first six.

In [ ]:
import hashlib
import math
import shutil
import tarfile
import time

import numpy as np
import requests
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

RET_CHUNK = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_e5_tok = AutoTokenizer.from_pretrained(E5_MODEL)
_e5 = AutoModel.from_pretrained(E5_MODEL, dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()


@torch.no_grad()
def e5_embed(texts, prefix, bs=128, max_len=192):
    out = []
    for i in range(0, len(texts), bs):
        b = _e5_tok([prefix + t for t in texts[i:i + bs]], padding=True, truncation=True,
                    max_length=max_len, return_tensors="pt").to(DEVICE)
        h = _e5(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
        v = (h * m).sum(1) / m.sum(1)
        out.append(torch.nn.functional.normalize(v.float(), dim=-1).cpu())
    return torch.cat(out).numpy()


def parse_raw(raw):
    """Title and text of a Pyserini document; the raw format differs between versions."""
    try:
        d = json.loads(raw)
    except Exception:
        d = {"contents": raw}
    if "title" in d and "text" in d:
        return d["title"], d["text"]
    c = d.get("contents", "")
    if "\n" in c:
        t, x = c.split("\n", 1)
        return t.strip().strip('"'), x.strip()
    return "", c


# ---- BM25 index: resumable download, MD5 check, unpacking to local disk ----
INDEX_NAME = "wikipedia-dpr-100w"
TAR_DIR = os.path.join(os.environ["PYSERINI_CACHE"], "tarballs")
LOCAL_INDEX_ROOT = "/content/bm25_index"
_MB = dict(unit="B", unit_scale=True, unit_divisor=1024)


def _download_resumable(url, dest):
    part = dest + ".part"
    have = os.path.getsize(part) if os.path.exists(part) else 0
    headers = {"User-Agent": "Mozilla/5.0"}
    if have:
        headers["Range"] = f"bytes={have}-"
    with requests.get(url, headers=headers, stream=True, timeout=60) as r:
        if have and r.status_code == 200:      # server ignored the range request
            have = 0
        r.raise_for_status()
        length = int(r.headers.get("Content-Length", 0))
        total = (have + length) if length else None
        with open(part, "ab" if have else "wb") as f, \
                tqdm(total=total, initial=have, desc="index download", **_MB) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    os.replace(part, dest)


def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(16 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


class _ProgressReader:
    def __init__(self, f, bar):
        self.f, self.bar = f, bar

    def read(self, n=-1):
        b = self.f.read(n)
        self.bar.update(len(b))
        return b


def ensure_bm25_index():
    from pyserini.prebuilt_index_info import TF_INDEX_INFO
    info = TF_INDEX_INFO[INDEX_NAME]
    os.makedirs(TAR_DIR, exist_ok=True)
    tar_path = os.path.join(TAR_DIR, info["filename"])
    ok_flag = tar_path + ".ok"
    if not os.path.exists(ok_flag):
        if not os.path.exists(tar_path):
            err = None
            for url in info["urls"]:
                try:
                    _download_resumable(url, tar_path)
                    err = None
                    break
                except Exception as e:
                    err = e
                    print(f"download failed ({url}): {e}")
            if err is not None:
                raise RuntimeError(f"index download failed: {err}. Rerun the cell to resume.")
        if _md5(tar_path) != info["md5"]:
            os.remove(tar_path)
            raise RuntimeError("MD5 mismatch; the file was removed. Rerun the cell.")
        open(ok_flag, "w").close()
    out_dir = os.path.join(LOCAL_INDEX_ROOT, info["md5"])
    done_flag = os.path.join(out_dir, ".extracted")
    if not os.path.exists(done_flag):
        shutil.rmtree(out_dir, ignore_errors=True)
        os.makedirs(out_dir)
        with open(tar_path, "rb") as raw, \
                tqdm(total=os.path.getsize(tar_path), desc="index unpacking", **_MB) as bar:
            with tarfile.open(fileobj=_ProgressReader(raw, bar), mode="r|*") as tf:
                for m in tf:
                    try:
                        tf.extract(m, out_dir, filter="data")
                    except TypeError:            # Python < 3.12
                        tf.extract(m, out_dir)
        open(done_flag, "w").close()
    for root, _, files in os.walk(out_dir):
        if any(f.startswith("segments_") for f in files):
            return root
    raise RuntimeError("Lucene index folder not found")


_searcher = None


def get_searcher():
    global _searcher
    if _searcher is None:
        from pyserini.search.lucene import LuceneSearcher
        _searcher = LuceneSearcher(ensure_bm25_index())
    return _searcher


def _retrieve_chunk(sub):
    qs, ids = sub["question"].tolist(), sub["id"].tolist()
    s = get_searcher()
    res = s.batch_search(qs, ids, k=BM25_K, threads=8)
    cand = {}
    for qid in ids:
        docs = []
        for h in res.get(qid, []):
            raw = h.raw if getattr(h, "raw", None) else s.doc(h.docid).raw()
            title, text = parse_raw(raw)
            docs.append({"docid": h.docid, "title": title, "text": text, "bm25": float(h.score)})
        cand[qid] = docs
    qv = e5_embed(qs, "query: ")
    rows = []
    for i, qid in enumerate(ids):
        docs = cand[qid]
        if not docs:
            rows.append({"id": qid, "ctxs": [], "phi": [float("nan")] * 7})
            continue
        dv = e5_embed([f"{d['title']} {d['text']}" for d in docs], "passage: ", bs=64)
        sim = dv @ qv[i]
        order = np.argsort(-sim)[:TOP_K]
        top = [dict(docs[j], e5=float(sim[j])) for j in order]
        bm = np.array(sorted([d["bm25"] for d in docs], reverse=True))
        es = np.sort(sim)[::-1]
        phi = [bm[0], bm[0] - (bm[1] if len(bm) > 1 else bm[0]),
               es[0], es[0] - (es[1] if len(es) > 1 else es[0]),
               float(np.mean(es[:TOP_K])), float(np.std(es[:TOP_K])), float(len(docs))]
        rows.append({"id": qid, "ctxs": top, "phi": [float(x) for x in phi]})
    return rows


def retrieve_dataset(name, df):
    out = f"{DIRS['retrieval']}/{name}_{N_PER_DATASET}_{SEED}_pyserini.jsonl"
    if os.path.exists(out):
        return pd.read_json(out, lines=True, dtype={"id": str})
    part_dir = out.replace(".jsonl", "_parts")
    os.makedirs(part_dir, exist_ok=True)
    n = len(df)
    n_chunks = math.ceil(n / RET_CHUNK)
    part = lambda c: f"{part_dir}/part_{c:05d}.jsonl"
    size = lambda c: min(RET_CHUNK, n - c * RET_CHUNK)
    todo = [c for c in range(n_chunks) if not os.path.exists(part(c))]
    with tqdm(total=n, initial=n - sum(size(c) for c in todo), desc=f"retrieval {name}", unit="q") as bar:
        for c in todo:
            sub = df.iloc[c * RET_CHUNK:(c + 1) * RET_CHUNK]
            rows = _retrieve_chunk(sub)
            tmp = part(c) + ".tmp"
            pd.DataFrame(rows).to_json(tmp, orient="records", lines=True, force_ascii=False)
            os.replace(tmp, part(c))
            bar.update(len(sub))
    r = pd.concat([pd.read_json(part(c), lines=True, dtype={"id": str}) for c in range(n_chunks)],
                  ignore_index=True)
    r.to_json(out, orient="records", lines=True, force_ascii=False)
    return r


RETRIEVAL = {name: retrieve_dataset(name, QUESTIONS[name]) for name in DATASETS}

## 7. Models and prompts

Models are loaded one at a time in bf16 without quantisation. Forward hooks on the stored layers keep the hidden
state of the last prompt token during the prefill step.

In [ ]:
import gc

from transformers import AutoModelForCausalLM

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
tok = model = LAYERS = None
LAYER_IDS, _hooks, _captured = [], [], {}
ACTIVE_MODEL = None


def get_layers(m):
    for path in ["model.layers", "model.language_model.layers", "transformer.h", "gpt_neox.layers"]:
        obj = m
        try:
            for p in path.split("."):
                obj = getattr(obj, p)
            return obj
        except AttributeError:
            continue
    raise RuntimeError("decoder layers not found")


def _make_hook(li):
    def hook(module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        if h.shape[1] > 1:                       # prefill step only
            _captured[li] = h[:, -1, :].detach().float().cpu()
    return hook


def free_model():
    global tok, model, LAYERS, _hooks, ACTIVE_MODEL
    for h in _hooks:
        h.remove()
    _hooks = []
    model = tok = LAYERS = None
    ACTIVE_MODEL = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def activate_model(name):
    global tok, model, LAYERS, LAYER_IDS, _hooks, ACTIVE_MODEL
    if ACTIVE_MODEL == name:
        return
    free_model()
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, device_map="auto",
                                                 low_cpu_mem_usage=True).eval()
    LAYERS = get_layers(model)
    n = len(LAYERS)
    LAYER_IDS = sorted({max(0, min(n - 1, round(f * n) - 1)) for f in LAYER_FRACS})
    _hooks = [LAYERS[li].register_forward_hook(_make_hook(li)) for li in LAYER_IDS]
    ACTIVE_MODEL = name
    print(f"{name}: {n} layers, stored {LAYER_IDS}, {DTYPE}")


SYS_NO = "Answer the question with a short answer of a few words. Do not explain."
SYS_RAG = ("Answer the question with a short answer of a few words. "
           "Use the documents if they are helpful. Do not explain.")


def build_prompt(q, ctxs=None):
    if ctxs:
        docs = "\n".join(f"[{i + 1}] ({c['title']}) {c['text']}" for i, c in enumerate(ctxs))
        msgs = [{"role": "system", "content": SYS_RAG},
                {"role": "user", "content": f"Documents:\n{docs}\n\nQuestion: {q}"}]
    else:
        msgs = [{"role": "system", "content": SYS_NO}, {"role": "user", "content": f"Question: {q}"}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:  # templates without a system role (Mistral): prepend the instruction to the user turn
        msgs = [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[-1]["content"]}]
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

## 8. Generation and features

Greedy decoding with at most 32 new tokens. For each answer the cache stores the text, the labels, the uncertainty
features (first-token entropy, mean and maximum token entropy, mean log-probability, first-token margin, number of
tokens) and the hidden states of the stored layers (float16).

In [ ]:
@torch.no_grad()
def generate_batch(prompts):
    _captured.clear()
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         return_dict_in_generate=True, output_scores=True,
                         pad_token_id=tok.pad_token_id)
    gen = out.sequences[:, enc["input_ids"].shape[1]:]
    texts = tok.batch_decode(gen, skip_special_tokens=True)

    logits = torch.stack(out.scores, dim=1).float()           # (B, T, V)
    logp = torch.log_softmax(logits, -1)
    ent = -(logp.exp() * logp).sum(-1)                        # (B, T)
    tok_lp = logp.gather(-1, gen.unsqueeze(-1)).squeeze(-1)   # (B, T)
    eos = getattr(model.generation_config, "eos_token_id", None)
    eos = list(eos) if isinstance(eos, (list, tuple)) else ([eos] if eos is not None else [])
    stop_ids = {tok.eos_token_id, tok.pad_token_id, *eos}
    mask = torch.ones_like(gen, dtype=torch.bool)             # tokens up to the first stop token
    for b in range(gen.shape[0]):
        for t in range(gen.shape[1]):
            if gen[b, t].item() in stop_ids:
                mask[b, t:] = False
                break
    mask[:, 0] = True
    m = mask.float()
    top2 = torch.topk(logp[:, 0].exp(), 2, dim=-1).values
    feats = {
        "ent_first": ent[:, 0].cpu().numpy(),
        "ent_mean": ((ent * m).sum(1) / m.sum(1)).cpu().numpy(),
        "ent_max": ent.masked_fill(~mask, -1).max(1).values.cpu().numpy(),
        "logp_mean": ((tok_lp * m).sum(1) / m.sum(1)).cpu().numpy(),
        "margin_first": (top2[:, 0] - top2[:, 1]).cpu().numpy(),
        "n_tokens": m.sum(1).cpu().numpy(),
    }
    H = np.stack([_captured[li].numpy() for li in LAYER_IDS], axis=1).astype(np.float16)  # (B, L, d)
    del out, logits, logp
    return texts, feats, H

## 9. Run the queue

Each pair is written in parts of 64 questions (`chunk_*.parquet` with the records, `chunk_*.npz` with the hidden
states). Finished pairs and the time per question go to `logs/run_status_<tag>.json`; this file is the source of the
compute-cost table.

In [ ]:
import datetime

STATUS_FILE = f"{DIRS['logs']}/run_status_{RUN_TAG}.json"


def load_status():
    return json.load(open(STATUS_FILE)) if os.path.exists(STATUS_FILE) else {}


def save_status(st):
    tmp = STATUS_FILE + ".tmp"
    json.dump(st, open(tmp, "w"), indent=1)
    os.replace(tmp, STATUS_FILE)


def chunk_starts(n):
    return list(range(0, n, CHUNK))


def is_done(out_dir, c0):
    return os.path.exists(f"{out_dir}/chunk_{c0:05d}.parquet") and os.path.exists(f"{out_dir}/chunk_{c0:05d}.npz")


def remaining(model_name, name):
    out_dir = f"{cache_dir(model_name)}/{name}"
    n = len(QUESTIONS[name])
    return sum(min(CHUNK, n - c0) for c0 in chunk_starts(n) if not is_done(out_dir, c0))


def run_dataset(name, out_dir, batch, qbar):
    qdf, rdf = QUESTIONS[name], RETRIEVAL[name].set_index("id")
    os.makedirs(out_dir, exist_ok=True)
    n = len(qdf)
    todo = [c0 for c0 in chunk_starts(n) if not is_done(out_dir, c0)]
    n_done = n - sum(min(CHUNK, n - c0) for c0 in todo)
    acc0 = acc1 = cnt = 0
    t0 = time.time()
    with tqdm(total=n, initial=n_done, desc=f"  {name}", unit="q", leave=False) as bar:
        for c0 in todo:
            part = qdf.iloc[c0:c0 + CHUNK]
            recs, H0s, H1s = [], [], []
            for b0 in range(0, len(part), batch):
                b = part.iloc[b0:b0 + batch]
                ctxs = [rdf.loc[i, "ctxs"] for i in b["id"]]
                t_no, f_no, H0 = generate_batch([build_prompt(q) for q in b["question"]])
                t_rag, f_rag, H1 = generate_batch([build_prompt(q, c) for q, c in zip(b["question"], ctxs)])
                H0s.append(H0)
                H1s.append(H1)
                for j, row in enumerate(b.itertuples()):
                    a0, em0, f10 = score_answer(t_no[j], row.golden_answers)
                    a1, em1, f11 = score_answer(t_rag[j], row.golden_answers)
                    rec = {"id": row.id, "dataset": name, "question": row.question,
                           "golden_answers": list(row.golden_answers), "pred0": t_no[j], "pred1": t_rag[j],
                           "y0": a0, "y1": a1, "em0": em0, "em1": em1, "f1_0": f10, "f1_1": f11,
                           "phi": rdf.loc[row.id, "phi"]}
                    for k, v in f_no.items():
                        rec[f"{k}_0"] = float(v[j])
                    for k, v in f_rag.items():
                        rec[f"{k}_1"] = float(v[j])
                    recs.append(rec)
                    acc0 += a0
                    acc1 += a1
                    cnt += 1
                bar.update(len(b))
                qbar.update(len(b))
                bar.set_postfix(closed_book=f"{acc0 / cnt:.2f}", rag=f"{acc1 / cnt:.2f}")
            tag = f"{out_dir}/chunk_{c0:05d}"
            np.savez_compressed(f"{tag}.tmp.npz", H0=np.concatenate(H0s), H1=np.concatenate(H1s),
                                layer_ids=np.array(LAYER_IDS))
            os.replace(f"{tag}.tmp.npz", f"{tag}.npz")
            pd.DataFrame(recs).to_parquet(f"{tag}.tmp.parquet")
            os.replace(f"{tag}.tmp.parquet", f"{tag}.parquet")
    return cnt, time.time() - t0


def run_queue():
    status = load_status()
    plan = [(m, d, remaining(m, d)) for m, ds in RUN_QUEUE for d in ds]
    total_left = sum(r for _, _, r in plan)
    print(f"{total_left} questions left in {sum(r > 0 for *_, r in plan)} pairs")
    with tqdm(total=total_left, desc="queue", unit="q") as qbar:
        for mname, dsets in RUN_QUEUE:
            todo = [d for d in dsets if remaining(mname, d) > 0]
            mtag = mname.split("/")[-1]
            if not todo:
                continue
            try:
                activate_model(mname)
            except Exception as e:
                msg = f"{type(e).__name__}: {str(e)[:300]}"
                print(f"\n{mtag} could not be loaded and is skipped: {msg}")
                status[f"{mtag}/_load_error"] = msg
                save_status(status)
                qbar.update(sum(remaining(mname, d) for d in todo))
                continue
            batch = GEN_BATCH_BY_MODEL.get(mname, GEN_BATCH)
            for d in todo:
                while True:
                    try:
                        cnt, sec = run_dataset(d, f"{cache_dir(mname)}/{d}", batch, qbar)
                        break
                    except torch.cuda.OutOfMemoryError:
                        _captured.clear()
                        gc.collect()
                        torch.cuda.empty_cache()
                        if batch <= 2:
                            raise
                        batch //= 2
                        print(f"\nout of memory, batch size for {mtag} is now {batch}")
                prev = status.get(f"{mtag}/{d}", {})
                status[f"{mtag}/{d}"] = {
                    "n": len(QUESTIONS[d]), "batch": batch,
                    "sec_per_q": round(sec / cnt, 4) if cnt else prev.get("sec_per_q"),
                    "finished": datetime.datetime.now().isoformat(timespec="seconds"),
                    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
                save_status(status)
                print(f"\n{mtag} / {d} done ({status[f'{mtag}/{d}']['sec_per_q']} s per question)")
    free_model()


run_queue()

## 10. Status

In [ ]:
st = load_status()
rows = []
for m, ds in RUN_QUEUE:
    mtag = m.split("/")[-1]
    for d in ds:
        left = remaining(m, d)
        state = "done" if left == 0 else ("load error" if f"{mtag}/_load_error" in st else "waiting")
        rows.append({"model": mtag, "dataset": d, "questions": len(QUESTIONS[d]), "left": left,
                     "state": state, "s/question": st.get(f"{mtag}/{d}", {}).get("sec_per_q")})
pd.DataFrame(rows)